# China customs exports — GACC vs SCI (and optional Kpler)

Reconcile **GACC customs-declared exports** (product × destination × month) against **SCI refined export shipping** and, optionally, **Kpler seaborne flows**.

**Data sources**
- GACC interactive export: `data/china_customs_data.csv` (manual download from [stats.customs.gov.cn](http://stats.customs.gov.cn/indexEn))
- SCI shipping actuals: `data/cache/latest/export_shipping.parquet`
- Kpler Flows (optional): requires `kpler` SDK + credentials in `.env`

**Desk products:** Gasoline, Jet, Gasoil/Diesel, FO

The residual gap (customs − AIS/shipping) is decomposed by **destination corridor** and **customs regime**, not reported as a single opaque number.

In [15]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fetch_data import load_cached_frames
from src.transforms import EXPORT_TABLE_PRODUCTS, build_export_monthly_from_shipping

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

GACC_CSV = ROOT / "data" / "china_customs_data.csv"
UNIT = "kt"
USE_KPLER = False  # flip when kpler SDK + KPLER_EMAIL/PASSWORD are available
LAG_MONTHS = (0, -1, 1)  # timing sensitivity for SCI vs customs month

print("Project root:", ROOT)
print("GACC file exists:", GACC_CSV.exists())

Project root: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\sci_api
GACC file exists: True


## 1. Product & destination maps

GACC reports 8-digit HS codes. We roll them into the same four desk buckets used in the SCI exports tab.

Destination names differ across sources; we normalize to a **canonical** label before joining.

In [16]:
HS_TO_PRODUCT = {
    "27101210": "Gasoline",
    "27101911": "Jet",
    "27101923": "Gasoil/Diesel",
    "27101922": "FO",
    "27101929": "FO",
    "27101924": "FO"
}

# GACC / SCI name variants → canonical destination (for joins & charts)
DESTINATION_ALIASES = {
    "hong kong": "Hong Kong",
    "hong kong, china": "Hong Kong",
    "hong kong,china": "Hong Kong",
    "hong kong£¬china": "Hong Kong",
    "hong kong of china": "Hong Kong",
    "macao": "Macao",
    "macao, china": "Macao",
    "macao,china": "Macao",
    "taiwan, china": "Taiwan Area",
    "taiwan,china": "Taiwan Area",
    "taiwan£¬china": "Taiwan Area",
    "taiwan area": "Taiwan Area",
    "republic of korea": "South Korea",
    "south korea": "South Korea",
    "democratic people's republic of korea": "North Korea",
    "viet nam": "Vietnam",
    "vietnam": "Vietnam",
    "united states of america": "U.S.",
    "united states": "U.S.",
    "u.s.": "U.S.",
    "usa": "U.S.",
    "united kingdom": "United Kingdom",
    "u.k.": "United Kingdom",
    "türkiye": "Turkey",
    "t¨¹rkiye": "Turkey",
    "turkey": "Turkey",
    "united arab emirates": "UAE",
    "uae": "UAE",
    "philippines": "Philippines",
    "singapore": "Singapore",
    "singapore republic": "Singapore",
    "malaysia": "Malaysia",
    "indonesia": "Indonesia",
    "japan": "Japan",
    "thailand": "Thailand",
    "mongolia": "Mongolia",
    "kazakhstan": "Kazakhstan",
    "myanmar": "Myanmar",
    "netherlands": "Netherlands",
    "australia": "Australia",
}


def canonical_destination(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    cleaned = str(name).strip()
    # GACC CSV uses fullwidth comma (U+FF0C) between place and "China".
    cleaned = re.sub(r"[£\uff0c，]+", ",", cleaned)
    cleaned = re.sub(r",\s*", ", ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    key = cleaned.lower()
    return DESTINATION_ALIASES.get(key, cleaned)


OVERLAND_HINTS = {
    "Mongolia", "Kazakhstan", "North Korea", "Myanmar", "Russia",
}
BUNKER_HUB_HINTS = {"Singapore", "Hong Kong", "Macao", "UAE"}


def corridor_bucket(destination: str) -> str:
    if destination in OVERLAND_HINTS:
        return "overland_hint"
    if destination in BUNKER_HUB_HINTS:
        return "bunker_transit_hint"
    return "seaborne_typical"

## 2. Load & normalize GACC customs data

In [17]:
def load_gacc_exports(path: Path = GACC_CSV) -> pd.DataFrame:
    raw = pd.read_csv(path, encoding="gbk")
    raw.columns = [c.strip().rstrip(",") for c in raw.columns]

    out = raw.rename(
        columns={
            "Date of data": "month",
            "Commodity code": "hs_code",
            "Commodity": "hs_description",
            "Trading partner code": "partner_code",
            "Trading partner": "destination_raw",
            "Customs Regime code": "regime_code",
            "Customs Regime": "regime",
            "Quantity": "quantity_kg",
            "Unit": "unit",
            "Renminbi Yuan": "value_rmb",
        }
    )

    out["hs_code"] = out["hs_code"].astype(str)
    out["product"] = out["hs_code"].map(HS_TO_PRODUCT)
    out = out.dropna(subset=["product"])

    out["month_start"] = pd.to_datetime(out["month"].astype(str), format="%Y%m")
    out["quantity_kg"] = pd.to_numeric(out["quantity_kg"], errors="coerce")
    out["quantity_kt"] = out["quantity_kg"] / 1_000_000
    if "value_rmb" in out.columns:
        out["value_rmb"] = (
            out["value_rmb"].astype(str).str.replace(",", "", regex=False)
        )
        out["value_rmb"] = pd.to_numeric(out["value_rmb"], errors="coerce")

    out["destination"] = out["destination_raw"].map(canonical_destination)
    return out


gacc = load_gacc_exports()
print(f"GACC rows: {len(gacc):,}")
print(
    "Months:", gacc["month_start"].min().date(), "→", gacc["month_start"].max().date(),
    f"({gacc['month_start'].nunique()} unique)"
)
print("Products:", gacc.groupby("product")["quantity_kt"].sum().round(1).to_dict())
gacc.head(3)

GACC rows: 11,030
Months: 2021-01-01 → 2026-05-01 (65 unique)
Products: {'FO': 78300.6, 'Gasoil/Diesel': 56600.6, 'Gasoline': 58364.2, 'Jet': 75965.7}


,month,hs_code,hs_description,partner_code,destination_raw,regime_code,regime,quantity_kg,unit,Supplementary Quantity,Supplementary Unit,value_rmb,product,month_start,quantity_kt,destination
0,202101,27101210,"Motor/aviation gasoline, without biodiesel",106,Myanmar,10,Ordinary Trade,125013,Kilogram,173518,Litre,415184,Gasoline,2021-01-01,0.125013,Myanmar
1,202101,27101210,"Motor/aviation gasoline, without biodiesel",106,Myanmar,14,Processing and Assembling,2300308,Kilogram,3192830,Litre,6663985,Gasoline,2021-01-01,2.300308,Myanmar
2,202101,27101210,"Motor/aviation gasoline, without biodiesel",112,Indonesia,10,Ordinary Trade,115179685,Kilogram,159869402,Litre,342218475,Gasoline,2021-01-01,115.179685,Indonesia


## 3. Load SCI export shipping (destination-level)

In [18]:
frames = load_cached_frames()
if frames is None or "export_shipping" not in frames:
    raise FileNotFoundError(
        "No SCI cache found. Run: python fetch_data.py --history"
    )

shipping = frames["export_shipping"].copy()
shipping["destination_raw"] = shipping["destination"]
shipping["destination"] = shipping["destination_raw"].map(canonical_destination)

sci_dest = (
    shipping.groupby(["month_start", "product", "destination"], as_index=False)["volume_kt"]
    .sum()
    .rename(columns={"volume_kt": "sci_kt"})
)

sci_monthly = build_export_monthly_from_shipping(shipping)

print(f"SCI shipping rows: {len(shipping):,}")
print(
    "Departure months:", shipping["month_start"].min().date(),
    "→", shipping["month_start"].max().date(),
)
print("Products:", shipping.groupby("product")["volume_kt"].sum().round(1).to_dict())
sci_dest.head(3)

SCI shipping rows: 4,114
Departure months: 2021-06-01 → 2026-06-01
Products: {'Gasoil/Diesel': 32712.9, 'Gasoline': 50295.8, 'Jet': 43960.6}


,month_start,product,destination,sci_kt
0,2021-06-01,Gasoil/Diesel,Australia,92.0
1,2021-06-01,Gasoil/Diesel,Hong Kong,125.0
2,2021-06-01,Gasoil/Diesel,Malaysia,18.0


## 4. Build reconciliation frame (month × product × destination)

In [19]:
def aggregate_gacc(
    df: pd.DataFrame,
    *,
    by_regime: bool = False,
) -> pd.DataFrame:
    keys = ["month_start", "product", "destination"]
    if by_regime:
        keys.append("regime")
    return (
        df.groupby(keys, as_index=False)
        .agg(gacc_kt=("quantity_kt", "sum"), value_rmb=("value_rmb", "sum"))
    )


gacc_dest = aggregate_gacc(gacc)
gacc_regime = aggregate_gacc(gacc, by_regime=True)

recon = gacc_dest.merge(
    sci_dest,
    on=["month_start", "product", "destination"],
    how="outer",
).fillna({"gacc_kt": 0.0, "sci_kt": 0.0})

recon["gap_sci"] = recon["gacc_kt"] - recon["sci_kt"]
recon["gap_sci_pct"] = np.where(
    recon["gacc_kt"] > 0,
    recon["gap_sci"] / recon["gacc_kt"] * 100,
    np.nan,
)
recon["corridor_bucket"] = recon["destination"].map(corridor_bucket)
recon["month"] = recon["month_start"].dt.strftime("%Y-%m")


def latest_shared_month(
    recon_df: pd.DataFrame,
    products: list[str],
) -> pd.Timestamp | None:
    """Latest month where every listed product has both GACC and SCI > 0."""
    month_sets: list[set[pd.Timestamp]] = []
    for product in products:
        sub = recon_df[recon_df["product"] == product]
        both = set(
            sub[(sub["gacc_kt"] > 0) & (sub["sci_kt"] > 0)]["month_start"].tolist()
        )
        month_sets.append(both)
    if not month_sets:
        return None
    common = set.intersection(*month_sets)
    if common:
        return pd.Timestamp(max(common))
    any_data = recon_df[recon_df["product"].isin(products)]
    any_data = any_data[(any_data["gacc_kt"] > 0) | (any_data["sci_kt"] > 0)]
    if any_data.empty:
        return None
    return pd.Timestamp(any_data["month_start"].max())


overlap = recon[(recon["gacc_kt"] > 0) & (recon["sci_kt"] > 0)]
print(f"Reconciliation rows: {len(recon):,}")
print(f"Rows with both GACC and SCI > 0: {len(overlap):,}")
if overlap.empty:
    print(
        "\n⚠ No overlapping months in current files. "
        "Append more years to china_customs_data.csv to join with SCI (2021+)."
    )
else:
    display(overlap.sort_values("gap_sci", ascending=False).head(15))

Reconciliation rows: 8,487
Rows with both GACC and SCI > 0: 951


,month_start,product,destination,gacc_kt,value_rmb,sci_kt,gap_sci,gap_sci_pct,corridor_bucket,month
4421,2023-05-01,Gasoline,Singapore,975.209701,5.127283e+09,405.0,570.209701,58.470471,bunker_transit_hint,2023-05
2348,2022-03-01,Gasoline,Pakistan,573.722464,3.482439e+09,42.0,531.722464,92.679387,seaborne_typical,2022-03
5863,2024-03-01,Gasoline,Singapore,975.329616,5.671626e+09,475.0,500.329616,51.298516,bunker_transit_hint,2024-03
3110,2022-08-01,Gasoline,Singapore,741.002965,5.511395e+09,267.0,474.002965,63.967756,bunker_transit_hint,2022-08
939,2021-06-01,Gasoline,Singapore,627.316443,2.483932e+09,171.0,456.316443,72.741030,bunker_transit_hint,2021-06
3748,2022-12-01,Jet,Netherlands,463.640866,3.113848e+09,25.0,438.640866,94.607895,seaborne_typical,2022-12
1579,2021-10-01,Gasoline,Pakistan,461.020772,2.153217e+09,40.0,421.020772,91.323601,seaborne_typical,2021-10
4819,2023-08-01,Gasoil/Diesel,Philippines,465.075188,2.536552e+09,90.0,375.075188,80.648290,seaborne_typical,2023-08
7512,2025-04-01,Gasoline,Singapore,675.203275,3.371492e+09,305.0,370.203275,54.828418,bunker_transit_hint,2025-04
3264,2022-09-01,Jet,Hong Kong,571.164728,4.055661e+09,213.0,358.164728,62.707781,bunker_transit_hint,2022-09


## 5. Product totals — GACC vs SCI (monthly wide)

In [20]:
gacc_monthly = (
    gacc.groupby(["month_start", "product"], as_index=False)["quantity_kt"]
    .sum()
    .rename(columns={"quantity_kt": "gacc_kt"})
)
gacc_wide = gacc_monthly.pivot(index="month_start", columns="product", values="gacc_kt").fillna(0)
gacc_wide = gacc_wide.reindex(columns=EXPORT_TABLE_PRODUCTS, fill_value=0)
gacc_wide.index = gacc_wide.index.strftime("%Y-%m")
gacc_wide.index.name = "month"

sci_wide = sci_monthly.set_index("month")[EXPORT_TABLE_PRODUCTS]
sci_wide.index.name = "month"

common_months = gacc_wide.index.intersection(sci_wide.index)

# Join on month index — assigning Series into a RangeIndex DataFrame misaligns and yields NaN.
gacc_part = gacc_wide.add_prefix("gacc_")
sci_part = sci_wide.add_prefix("sci_")
totals = gacc_part.join(sci_part, how="outer")

for prod in EXPORT_TABLE_PRODUCTS:
    if prod == "FO":
        continue  # SCI does not report FO
    totals[f"gap_{prod}"] = totals[f"gacc_{prod}"].fillna(0) - totals[f"sci_{prod}"].fillna(0)

print("GACC monthly totals (kt):")
display(gacc_wide.round(1))

if len(common_months):
    gap_cols = [f"gap_{p}" for p in EXPORT_TABLE_PRODUCTS if p != "FO"]
    show_cols = (
        [f"gacc_{p}" for p in EXPORT_TABLE_PRODUCTS if p != "FO"]
        + [f"sci_{p}" for p in EXPORT_TABLE_PRODUCTS if p != "FO"]
        + gap_cols
    )
    print(f"\nOverlapping months for product-level gap: {len(common_months)} ({common_months.min()} → {common_months.max()})")
    display(totals.loc[common_months, show_cols].round(1))
else:
    print("\nNo overlapping months yet — GACC and SCI periods do not intersect.")

GACC monthly totals (kt):


product,Gasoline,Jet,Gasoil/Diesel,FO
month,,,,
2021-01,1890.3,361.8,1721.1,1676.7
2021-02,1648.1,515.3,1716.0,1397.7
2021-03,1553.9,715.2,2809.9,1713.9
2021-04,1469.7,658.4,2719.9,1934.3
2021-05,1544.5,565.1,1680.9,1580.8
...,...,...,...,...
2026-01,530.4,0.0,0.0,0.0
2026-02,318.0,0.0,0.0,0.0
2026-03,300.7,0.0,0.0,0.0



Overlapping months for product-level gap: 60 (2021-06 → 2026-05)


product,gacc_Gasoline,gacc_Jet,gacc_Gasoil/Diesel,sci_Gasoline,sci_Jet,sci_Gasoil/Diesel,gap_Gasoline,gap_Jet,gap_Gasoil/Diesel
month,,,,,,,,,
2021-06,1452.4,910.2,2362.3,391.0,205.0,628.0,1061.4,705.2,1734.3
2021-07,740.0,661.7,1392.0,289.0,134.0,593.0,451.0,527.7,799.0
2021-08,566.1,920.8,546.2,328.0,730.0,477.0,238.1,190.8,69.2
2021-09,922.1,886.0,776.9,362.0,210.0,424.0,560.1,676.0,352.9
2021-10,1009.1,830.0,555.6,563.0,244.0,120.0,446.1,586.0,435.6
2021-11,807.4,942.2,599.1,483.4,62.0,10.0,324.0,880.2,589.1
2021-12,938.5,591.4,325.2,583.7,419.7,214.9,354.8,171.7,110.3
2022-01,861.1,555.3,213.1,743.0,501.0,83.0,118.1,54.3,130.1
2022-02,1025.9,670.5,197.6,1082.0,344.0,99.0,-56.1,326.5,98.6


## 6. Gap decomposition by corridor & customs regime

In [21]:
def gap_summary(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    out = (
        df.groupby(group_cols, as_index=False)
        .agg(
            gacc_kt=("gacc_kt", "sum"),
            sci_kt=("sci_kt", "sum"),
            gap_sci=("gap_sci", "sum"),
        )
    )
    out["gap_share_of_gacc"] = np.where(
        out["gacc_kt"] > 0, out["gap_sci"] / out["gacc_kt"] * 100, np.nan
    )
    return out.sort_values("gap_sci", ascending=False)


# Corridor-level gap (all GACC months — SCI zero where no overlap)
by_corridor = gap_summary(recon, ["product", "corridor_bucket"])
print("Gap by corridor bucket (all data in CSV):")
display(by_corridor.round(1))

# Top destination corridors per product
by_dest = gap_summary(recon, ["product", "destination", "corridor_bucket"])
print("\nTop 15 destination gaps (GACC − SCI):")
display(by_dest.head(15).round(1))

# Customs regime explains non-seaborne / bonded flows
regime_totals = (
    gacc_regime.groupby(["product", "regime"], as_index=False)["gacc_kt"]
    .sum()
    .sort_values(["product", "gacc_kt"], ascending=[True, False])
)
print("\nGACC volume by customs regime:")
display(regime_totals.round(1))

Gap by corridor bucket (all data in CSV):


,product,corridor_bucket,gacc_kt,sci_kt,gap_sci,gap_share_of_gacc
2,FO,seaborne_typical,52170.8,0.0,52170.8,100.0
0,FO,bunker_transit_hint,26055.0,0.0,26055.0,100.0
11,Jet,seaborne_typical,48101.9,23057.0,25044.9,52.1
5,Gasoil/Diesel,seaborne_typical,42337.2,26563.0,15774.2,37.3
6,Gasoline,bunker_transit_hint,35309.6,27253.0,8056.6,22.8
3,Gasoil/Diesel,bunker_transit_hint,13172.8,6139.9,7032.9,53.4
9,Jet,bunker_transit_hint,27225.9,20903.6,6322.3,23.2
4,Gasoil/Diesel,overland_hint,1090.5,10.0,1080.5,99.1
10,Jet,overland_hint,637.9,0.0,637.9,100.0
7,Gasoline,overland_hint,579.9,20.0,559.9,96.6



Top 15 destination gaps (GACC − SCI):


,product,destination,corridor_bucket,gacc_kt,sci_kt,gap_sci,gap_share_of_gacc
46,FO,Hong Kong,bunker_transit_hint,19259.1,0.0,19259.1,100.0
62,FO,Liberia,seaborne_typical,15051.5,0.0,15051.5,100.0
87,FO,Panama,seaborne_typical,13011.7,0.0,13011.7,100.0
69,FO,Marshall Islands,seaborne_typical,7727.9,0.0,7727.9,100.0
281,Gasoline,Singapore,bunker_transit_hint,33399.4,26010.6,7388.8,22.1
136,Gasoil/Diesel,Bangladesh,seaborne_typical,7955.1,802.0,7153.1,89.9
104,FO,Singapore,bunker_transit_hint,6785.8,0.0,6785.8,100.0
366,Jet,Netherlands,seaborne_typical,5518.0,1396.3,4121.7,74.7
223,Gasoil/Diesel,Singapore,bunker_transit_hint,7057.8,3154.9,3902.9,55.3
277,Gasoline,Pakistan,seaborne_typical,4749.6,1467.8,3281.8,69.1



GACC volume by customs regime:


,product,regime,gacc_kt
0,FO,Customs Warehousing Trade,78054.1
2,FO,Processing and Assembling,155.7
3,FO,Processing with Imported Materials,57.6
1,FO,Ordinary Trade,33.2
7,Gasoil/Diesel,Ordinary Trade,50485.5
6,Gasoil/Diesel,Customs Warehousing Trade,4683.1
8,Gasoil/Diesel,Processing and Assembling,1115.6
5,Gasoil/Diesel,Border Trade,225.0
9,Gasoil/Diesel,Processing with Imported Materials,81.0
4,Gasoil/Diesel,Aid or Donation Between Governments or by Inte...,10.5


## 7. Charts

In [22]:
# --- 7a. GACC monthly exports by product ---
plot_df = gacc_wide.reset_index().melt(id_vars="month", var_name="product", value_name="gacc_kt")
plot_df["month"] = pd.Categorical(plot_df["month"], categories=gacc_wide.index.tolist(), ordered=True)
fig_products = px.line(
    plot_df,
    x="month",
    y="gacc_kt",
    color="product",
    markers=True,
    title="GACC customs exports by product (kt)",
    labels={"gacc_kt": "Volume (kt)", "month": "Month"},
)
fig_products.update_layout(
    legend_title_text="Product",
    xaxis=dict(type="category", tickangle=-45),
    hovermode="x unified",
)
fig_products.show()

# --- 7c. Top destinations for a selected product/month ---
FOCUS_PRODUCT = "Gasoil/Diesel"

product_monthly = (
    gacc[gacc["product"] == FOCUS_PRODUCT]
    .groupby("month_start", as_index=False)["quantity_kt"]
    .sum()
)
# Latest month with data for this product (max month overall may lack diesel in partial 2026 pulls).
months_with_data = product_monthly.loc[product_monthly["quantity_kt"] > 0, "month_start"]
FOCUS_MONTH = months_with_data.max() if not months_with_data.empty else gacc["month_start"].max()

dest_month = (
    gacc[(gacc["product"] == FOCUS_PRODUCT) & (gacc["month_start"] == FOCUS_MONTH)]
    .groupby("destination", as_index=False)["quantity_kt"]
    .sum()
    .sort_values("quantity_kt", ascending=True)
    .tail(15)
)

if dest_month.empty:
    print(f"No {FOCUS_PRODUCT} destination data for {FOCUS_MONTH:%Y-%m}.")
else:
    fig_dest = px.bar(
        dest_month,
        x="quantity_kt",
        y="destination",
        orientation="h",
        title=f"GACC {FOCUS_PRODUCT} exports — top destinations ({FOCUS_MONTH:%Y-%m}, kt)",
    )
    fig_dest.show()

# --- 7d. Regime mix ---
regime_plot = (
    gacc.groupby(["month_start", "regime"], as_index=False)["quantity_kt"]
    .sum()
)
regime_plot["month"] = regime_plot["month_start"].dt.strftime("%Y-%m")
fig_regime = px.area(
    regime_plot,
    x="month",
    y="quantity_kt",
    color="regime",
    title="GACC exports by customs regime (all products, kt)",
)
fig_regime.show()

### 7b. Destination gap maps (Folium)

Flow lines from China to each destination, coloured by **GACC − SCI** (kt). One map per product — red = customs above tracker, blue = SCI above customs.

**If the map is blank in Cursor/VS Code:** the cell saves HTML under `notebooks/_map_output/` — open that file in your browser. You can also run **Notebook: Trust** (command palette) so inline maps load.

In [23]:
import folium
from folium import plugins
from IPython.display import HTML, display

from src.gacc_geo import CHINA_ORIGIN, coords_for_destination

MAP_OUTPUT_DIR = ROOT / "notebooks" / "_map_output"
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def show_folium_map(fmap: folium.Map, *, product: str, month: pd.Timestamp) -> Path:
    """Inline HTML (no iframe) + saved file for Cursor/VS Code notebook viewers."""
    # Direct Leaflet embed — works without "Trust Notebook" unlike display(fmap).
    display(HTML(fmap.get_root().render()))

    slug = f"{month:%Y-%m}_{product}".replace("/", "-").replace(" ", "_")
    out_path = MAP_OUTPUT_DIR / f"gap_map_{slug}.html"
    fmap.save(str(out_path))
    print(f"Saved: {out_path}  (open in browser if inline map is blank)")
    return out_path


def destination_gap_folium_map(
    recon_df: pd.DataFrame,
    *,
    product: str,
    month_start: pd.Timestamp,
    min_abs_gap_kt: float = 5.0,
    max_flows: int = 40,
) -> folium.Map | None:
    """Build a Folium map of export corridor gaps for one product-month."""
    sub = recon_df[
        (recon_df["product"] == product) & (recon_df["month_start"] == month_start)
    ].copy()
    sub = sub[(sub["gacc_kt"] > 0) | (sub["sci_kt"] > 0)]
    if sub.empty:
        return None

    sub["gap_sci"] = sub["gacc_kt"] - sub["sci_kt"]
    sub["abs_gap"] = sub["gap_sci"].abs()
    sub = sub[sub["abs_gap"] >= min_abs_gap_kt].nlargest(max_flows, "abs_gap")
    if sub.empty:
        return None

    def _dest_coords(dest: str) -> pd.Series:
        coords = coords_for_destination(dest)
        if coords is None:
            return pd.Series({"lat": np.nan, "lon": np.nan})
        return pd.Series({"lat": coords[0], "lon": coords[1]})

    sub = sub.join(sub["destination"].apply(_dest_coords))
    mapped = sub.dropna(subset=["lat", "lon"])
    if mapped.empty:
        return None

    fmap = folium.Map(location=list(CHINA_ORIGIN), zoom_start=3, tiles="CartoDB positron")
    folium.Marker(
        CHINA_ORIGIN,
        popup=folium.Popup("<b>China</b><br>export origin", max_width=200),
        icon=folium.Icon(color="green", icon="play"),
    ).add_to(fmap)

    max_gap = float(mapped["abs_gap"].max()) or 1.0
    for row in mapped.itertuples():
        color = "#d62728" if row.gap_sci >= 0 else "#1f77b4"
        weight = 1.5 + 5.0 * (row.abs_gap / max_gap)
        radius = 4 + 14.0 * (row.abs_gap / max_gap)
        folium.PolyLine(
            locations=[CHINA_ORIGIN, (row.lat, row.lon)],
            color=color,
            weight=weight,
            opacity=0.65,
            tooltip=(
                f"{row.destination}: GACC {row.gacc_kt:,.0f} · SCI {row.sci_kt:,.0f} · "
                f"gap {row.gap_sci:+,.0f} kt"
            ),
        ).add_to(fmap)
        folium.CircleMarker(
            location=(row.lat, row.lon),
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
            popup=folium.Popup(
                (
                    f"<b>{row.destination}</b><br>"
                    f"GACC: {row.gacc_kt:,.1f} kt<br>"
                    f"SCI: {row.sci_kt:,.1f} kt<br>"
                    f"Gap (GACC−SCI): {row.gap_sci:+,.1f} kt"
                ),
                max_width=280,
            ),
        ).add_to(fmap)

    plugins.MiniMap(toggle_display=True).add_to(fmap)
    title = f"{product} — GACC−SCI gap by destination ({month_start:%Y-%m}, kt)"
    fmap.get_root().html.add_child(folium.Element(f"<h4 style='margin:8px'>{title}</h4>"))
    return fmap


MAP_PRODUCTS: list[str] | None = None  # None → Gasoline, Jet, Gasoil/Diesel
MAP_MONTH: str | None = None  # e.g. "2025-12"
MAP_MIN_ABS_GAP_KT = 5.0
MAP_MAX_FLOWS = 40

map_products = MAP_PRODUCTS or [p for p in EXPORT_TABLE_PRODUCTS if p != "FO"]
if MAP_MONTH:
    map_ts = pd.Timestamp(MAP_MONTH + "-01")
else:
    map_ts = latest_shared_month(recon, map_products)

if map_ts is None:
    print("No overlapping months for gap maps.")
else:
    print(f"Gap maps month: {map_ts:%Y-%m}")
    unmapped: list[str] = []
    for product in map_products:
        sub = recon[
            (recon["product"] == product)
            & (recon["month_start"] == map_ts)
            & ((recon["gacc_kt"] > 0) | (recon["sci_kt"] > 0))
        ]
        missing = [
            d
            for d in sub["destination"].unique()
            if coords_for_destination(d) is None and sub.loc[sub["destination"] == d, "gacc_kt"].sum()
            + sub.loc[sub["destination"] == d, "sci_kt"].sum()
            > 0
        ]
        unmapped.extend(missing)

        fmap = destination_gap_folium_map(
            recon,
            product=product,
            month_start=map_ts,
            min_abs_gap_kt=MAP_MIN_ABS_GAP_KT,
            max_flows=MAP_MAX_FLOWS,
        )
        if fmap is None:
            print(f"  {product}: no mappable gaps")
        else:
            show_folium_map(fmap, product=product, month=map_ts)

    if unmapped:
        print("Destinations without coordinates (add to src/gacc_geo.py):", sorted(set(unmapped))[:20])

Gap maps month: 2025-12


Saved: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\sci_api\notebooks\_map_output\gap_map_2025-12_Gasoline.html  (open in browser if inline map is blank)


Saved: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\sci_api\notebooks\_map_output\gap_map_2025-12_Jet.html  (open in browser if inline map is blank)


Saved: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\sci_api\notebooks\_map_output\gap_map_2025-12_Gasoil-Diesel.html  (open in browser if inline map is blank)


### 7c. Destination label audit

When the gap chart shows two similar names (e.g. `Hong Kong` and `Hong Kong,China`), they are usually the **same market** with different spelling — not two real destinations. Use this table to compare **raw labels**, **GACC partner codes**, and volumes before trusting the gap.

In [24]:
def audit_destination_labels(
    gacc_df: pd.DataFrame,
    shipping_df: pd.DataFrame,
    *,
    product: str,
    month_start: pd.Timestamp,
    search: str,
) -> pd.DataFrame:
    """Side-by-side raw labels, partner codes, and volumes for one market."""
    needle = search.casefold()

    g = gacc_df[
        (gacc_df["product"] == product)
        & (gacc_df["month_start"] == month_start)
        & (
            gacc_df["destination_raw"].str.casefold().str.contains(needle, na=False)
            | gacc_df["destination"].str.casefold().str.contains(needle, na=False)
        )
    ]
    g_side = (
        g.groupby(["partner_code", "destination_raw", "destination"], as_index=False)["quantity_kt"]
        .sum()
        .assign(source="GACC")
        .rename(columns={"destination_raw": "raw_label", "destination": "canonical"})
    )

    s = shipping_df[
        (shipping_df["product"] == product) & (shipping_df["month_start"] == month_start)
    ].copy()
    raw_col = "destination_raw" if "destination_raw" in s.columns else "destination"
    s["raw_label"] = s[raw_col]
    s["canonical"] = s["destination"]
    s = s[
        s["raw_label"].str.casefold().str.contains(needle, na=False)
        | s["canonical"].str.casefold().str.contains(needle, na=False)
    ]
    s_side = (
        s.groupby(["raw_label", "canonical"], as_index=False)["volume_kt"]
        .sum()
        .assign(source="SCI", partner_code=pd.NA)
        .rename(columns={"volume_kt": "quantity_kt"})
    )

    cols = ["source", "partner_code", "raw_label", "canonical", "quantity_kt"]
    return (
        pd.concat([g_side[cols], s_side[cols]], ignore_index=True)
        .sort_values(["source", "quantity_kt"], ascending=[True, False])
    )


AUDIT_PRODUCT = "Jet"
AUDIT_MONTH = pd.Timestamp("2025-12-01")
AUDIT_SEARCH = "hong"  # try "singap", "korea", etc.

print(f"Label audit — {AUDIT_PRODUCT} {AUDIT_MONTH:%Y-%m} — search={AUDIT_SEARCH!r}")
display(
    audit_destination_labels(
        gacc, shipping,
        product=AUDIT_PRODUCT,
        month_start=AUDIT_MONTH,
        search=AUDIT_SEARCH,
    ).round(1)
)

# Merged view: should be ONE row per canonical destination after alias fix.
merged = (
    gacc[(gacc["product"] == AUDIT_PRODUCT) & (gacc["month_start"] == AUDIT_MONTH)]
    .groupby("destination", as_index=False)["quantity_kt"].sum()
    .merge(
        shipping[(shipping["product"] == AUDIT_PRODUCT) & (shipping["month_start"] == AUDIT_MONTH)]
        .groupby("destination", as_index=False)["volume_kt"].sum(),
        on="destination",
        how="outer",
    )
    .fillna(0)
    .rename(columns={"quantity_kt": "gacc_kt", "volume_kt": "sci_kt"})
)
merged["gap_sci"] = merged["gacc_kt"] - merged["sci_kt"]
print("\nAfter canonical merge (filtered):")
display(merged[merged["destination"].str.contains(AUDIT_SEARCH, case=False, na=False)].round(1))

Label audit — Jet 2025-12 — search='hong'


,source,partner_code,raw_label,canonical,quantity_kt
0,GACC,110,Hong Kong，China,Hong Kong,806.8
1,SCI,<NA>,Hong Kong,Hong Kong,555.0



After canonical merge (filtered):


,destination,gacc_kt,sci_kt,gap_sci
18,Hong Kong,806.8,555.0,251.8


### 7d. Destination gap — GACC vs SCI

Horizontal bars show **GACC − SCI** (kt) by destination for a chosen month. Positive = customs above the tracker; negative = SCI above customs.

In [25]:
def plot_destination_gap(
    recon_df: pd.DataFrame,
    *,
    product: str,
    month_start: pd.Timestamp,
    top_n: int = 15,
) -> go.Figure | None:
    """Top destinations by |GACC − SCI| for one product-month."""
    sub = recon_df[
        (recon_df["product"] == product) & (recon_df["month_start"] == month_start)
    ].copy()
    sub = sub[(sub["gacc_kt"] > 0) | (sub["sci_kt"] > 0)]
    if sub.empty:
        return None

    sub["gap_sci"] = sub["gacc_kt"] - sub["sci_kt"]
    sub["abs_gap"] = sub["gap_sci"].abs()
    top = (
        sub.nlargest(top_n, "abs_gap")
        .sort_values("gap_sci", ascending=True)
    )
    top["tracker_vs_govt"] = np.where(
        top["gap_sci"] >= 0,
        "GACC > SCI (under-tracked)",
        "SCI > GACC (over-tracked)",
    )

    fig = px.bar(
        top,
        x="gap_sci",
        y="destination",
        orientation="h",
        color="tracker_vs_govt",
        color_discrete_map={
            "GACC > SCI (under-tracked)": "#d62728",
            "SCI > GACC (over-tracked)": "#1f77b4",
        },
        title=(
            f"{product} — destination gap GACC − SCI "
            f"({month_start:%Y-%m}, top {len(top)} by |gap|, kt)"
        ),
        labels={
            "gap_sci": "GACC − SCI (kt)",
            "destination": "Destination",
            "tracker_vs_govt": "",
        },
    )
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="gray")
    fig.update_layout(showlegend=True, legend_title_text="")
    return fig


def latest_shared_month(
    recon_df: pd.DataFrame,
    products: list[str],
) -> pd.Timestamp | None:
    """Latest month where every listed product has both GACC and SCI > 0."""
    month_sets: list[set[pd.Timestamp]] = []
    for product in products:
        sub = recon_df[recon_df["product"] == product]
        both = set(
            sub[(sub["gacc_kt"] > 0) & (sub["sci_kt"] > 0)]["month_start"].tolist()
        )
        month_sets.append(both)
    if not month_sets:
        return None
    common = set.intersection(*month_sets)
    if common:
        return pd.Timestamp(max(common))
    # Fall back: latest month with any GACC or SCI for any listed product.
    any_data = recon_df[recon_df["product"].isin(products)]
    any_data = any_data[(any_data["gacc_kt"] > 0) | (any_data["sci_kt"] > 0)]
    if any_data.empty:
        return None
    return pd.Timestamp(any_data["month_start"].max())


# Pass a list to compare multiple products; None uses all SCI-tracked desk buckets.
COMPARE_PRODUCTS: list[str] | None = None  # e.g. ["Gasoline", "Jet", "Gasoil/Diesel"]
COMPARE_MONTH: str | None = None  # e.g. "2025-12"; None = latest shared month
TOP_N_DESTINATIONS = 15

products_to_plot = COMPARE_PRODUCTS or [p for p in EXPORT_TABLE_PRODUCTS if p != "FO"]
if COMPARE_MONTH:
    compare_ts = pd.Timestamp(COMPARE_MONTH + "-01")
else:
    compare_ts = latest_shared_month(recon, products_to_plot)

if compare_ts is None:
    print("No overlapping GACC/SCI months for destination gap charts.")
else:
    print(f"Destination gap month: {compare_ts:%Y-%m}")
    for product in products_to_plot:
        fig_gap = plot_destination_gap(
            recon,
            product=product,
            month_start=compare_ts,
            top_n=TOP_N_DESTINATIONS,
        )
        if fig_gap is None:
            print(f"  {product}: no destination data for {compare_ts:%Y-%m}")
        else:
            fig_gap.show()

Destination gap month: 2025-12


## 8. Seasonality by calendar year

Monthly GACC export volumes by product — one line per calendar year (current year highlighted in red), matching the country demand dashboard style.

In [26]:
def gacc_seasonality_chart(
    df: pd.DataFrame,
    products: list[str],
    *,
    highlight_year: int | None = None,
    cols: int = 2,
) -> go.Figure:
    """Small-multiples seasonality: one panel per product, one line per calendar year."""
    work = df[df["product"].isin(products)].copy()
    work["year"] = work["month_start"].dt.year
    work["month_num"] = work["month_start"].dt.month
    work = (
        work.groupby(["product", "year", "month_num"], as_index=False)["quantity_kt"]
        .sum()
    )

    highlight_year = highlight_year or int(work["year"].max())
    years_sorted = sorted(work["year"].unique())
    palette = ("#90c2e7", "#b3a2c7", "#a8dba8", "#f4b183", "#bdbdbd")
    month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    rows = (len(products) + cols - 1) // cols
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=products,
        shared_xaxes=False,
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    seen_legend: set[int] = set()
    for idx, product in enumerate(products):
        r, c = idx // cols + 1, idx % cols + 1
        sub = work[work["product"] == product]
        for year_idx, year in enumerate(years_sorted):
            ydata = sub[sub["year"] == year].sort_values("month_num")
            if ydata.empty:
                continue
            is_highlight = year == highlight_year
            line = dict(
                color="#d62728" if is_highlight else palette[year_idx % len(palette)],
                width=3 if is_highlight else 1.4,
            )
            show_legend = year not in seen_legend
            if show_legend:
                seen_legend.add(year)
            fig.add_trace(
                go.Scatter(
                    x=ydata["month_num"],
                    y=ydata["quantity_kt"],
                    mode="lines+markers",
                    name=str(year),
                    legendgroup=str(year),
                    showlegend=show_legend,
                    line=line,
                    marker=dict(size=5),
                    hovertemplate=(
                        f"<b>{product}</b><br>Year: {year}<br>"
                        "Month: %{x}<br>Volume: %{y:,.1f} kt<extra></extra>"
                    ),
                ),
                row=r,
                col=c,
            )

    fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)), ticktext=month_labels)
    fig.update_yaxes(title_text="kt")
    year_min, year_max = int(work["year"].min()), int(work["year"].max())
    fig.update_layout(
        title=(
            f"GACC customs exports — seasonality by calendar year "
            f"({year_min}–{year_max}, {highlight_year} in red)"
        ),
        template="plotly_white",
        height=280 * rows,
        hovermode="closest",
        legend=dict(
            orientation="h",
            y=-0.08,
            yanchor="top",
            x=0.5,
            xanchor="center",
            title_text="Calendar year",
        ),
        margin=dict(t=70, l=60, r=20, b=70),
    )
    return fig


season_products = [p for p in EXPORT_TABLE_PRODUCTS if p in gacc["product"].unique()]
fig_season = gacc_seasonality_chart(gacc, season_products, cols=2)
fig_season.show()

## 9. Timing lag sensitivity (when months overlap)

Customs declaration month may not equal SCI **departure** month. Re-run the join with SCI shifted ±1 month.

In [27]:
def recon_with_lag(lag: int) -> pd.DataFrame:
    sci = sci_dest.copy()
    sci["month_start"] = sci["month_start"] + pd.DateOffset(months=lag)
    merged = gacc_dest.merge(
        sci,
        on=["month_start", "product", "destination"],
        how="inner",
    )
    merged["gap_sci"] = merged["gacc_kt"] - merged["sci_kt"]
    return merged


lag_rows = []
for lag in LAG_MONTHS:
    m = recon_with_lag(lag)
    if m.empty:
        continue
    lag_rows.append(
        {
            "sci_lag_months": lag,
            "matched_rows": len(m),
            "gacc_kt": m["gacc_kt"].sum(),
            "sci_kt": m["sci_kt"].sum(),
            "gap_sci": m["gap_sci"].sum(),
            "mae_kt": (m["gap_sci"].abs()).mean(),
        }
    )

if lag_rows:
    display(pd.DataFrame(lag_rows).round(1))
else:
    print("No overlapping months — lag analysis will run once GACC CSV covers 2021+.")

,sci_lag_months,matched_rows,gacc_kt,sci_kt,gap_sci,mae_kt
0,0,951,122428.9,106610.2,15818.7,62.2
1,-1,920,118846.3,105123.6,13722.7,74.4
2,1,908,117182.2,102590.6,14591.6,68.8


## 10. Optional — Kpler destination split

Set `USE_KPLER = True` and install the Kpler SDK. Pulls China origin exports with `FlowsSplit.DestinationCountries`.

In [28]:
KPLER_PRODUCTS = {
    "Gasoline": ["Gasoline", "95 Gasoline", "98 Gasoline"],
    "Jet": ["Jet", "Kerosene"],
    "Gasoil/Diesel": ["Diesel", "Gasoil"],
    "FO": ["Fuel Oils", "VLSFO", "HSFO"],
}


def fetch_kpler_dest_monthly(
    *,
    start: str = "2015-01",
    end: str | None = None,
) -> pd.DataFrame:
    from datetime import date

    import os
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
    from kpler.sdk import (
        Flows,
        FlowsConfig,
        FlowsDirection,
        FlowsMeasurementUnit,
        FlowsPeriod,
        FlowsSplit,
        Platform,
    )

    email = os.getenv("KPLER_EMAIL")
    password = os.getenv("KPLER_PASSWORD")
    if not email or not password:
        raise ValueError("Set KPLER_EMAIL and KPLER_PASSWORD in sci_api/.env")

    end = end or date.today().strftime("%Y-%m")
    start_date = pd.Timestamp(start + "-01").date()
    end_date = pd.Timestamp(end + "-01") + pd.offsets.MonthEnd(0)
    end_date = end_date.date()

    client = Flows(
        FlowsConfig(platform=Platform.Liquids, email=email, password=password)
    )

    rows = []
    for desk_product, api_products in KPLER_PRODUCTS.items():
        for api_product in api_products:
            df = client.get(
                flow_direction=[FlowsDirection.Export],
                split=[FlowsSplit.DestinationCountries],
                granularity=[FlowsPeriod.Monthly],
                products=[api_product],
                unit=[FlowsMeasurementUnit.T],
                from_zones=["China"],
                start_date=start_date,
                end_date=end_date,
                only_realized=True,
                with_intra_country=True,
            )
            if df.empty:
                continue
            value_cols = [c for c in df.columns if c not in ("Date",)]
            long = df.melt(id_vars=["Date"], value_vars=value_cols, var_name="destination", value_name="tonnes")
            long["month_start"] = pd.to_datetime(long["Date"]).dt.to_period("M").dt.to_timestamp()
            long["kpler_kt"] = pd.to_numeric(long["tonnes"], errors="coerce") / 1000
            long["product"] = desk_product
            long["destination"] = long["destination"].map(canonical_destination)
            rows.append(long[["month_start", "product", "destination", "kpler_kt"]])

    if not rows:
        return pd.DataFrame(columns=["month_start", "product", "destination", "kpler_kt"])

    out = pd.concat(rows, ignore_index=True)
    return out.groupby(["month_start", "product", "destination"], as_index=False)["kpler_kt"].sum()


if USE_KPLER:
    kpler_dest = fetch_kpler_dest_monthly()
    recon3 = recon.merge(kpler_dest, on=["month_start", "product", "destination"], how="outer").fillna(0)
    recon3["gap_kpler"] = recon3["gacc_kt"] - recon3["kpler_kt"]
    display(gap_summary(recon3, ["product", "destination"]).head(15).round(1))
else:
    print("Kpler disabled. Set USE_KPLER=True and add credentials to compare AIS seaborne flows.")

Kpler disabled. Set USE_KPLER=True and add credentials to compare AIS seaborne flows.


## 11. Next steps

- [ ] Append **2021+** GACC exports to `data/china_customs_data.csv` (same portal export format) to unlock SCI joins
- [ ] Extend `DESTINATION_ALIASES` for unmatched SCI / GACC labels
- [ ] Enable Kpler (`USE_KPLER=True`) for three-way corridor reconciliation
- [ ] Scraper for `stats.customs.gov.cn` (deferred)
- [ ] Validate GACC product totals against monthly bulletin Table (13) when needed